# Bike sharing demand -- StudioPhase 4 of the domain track. Same model as `train.ipynb`, run from aJupyterLab space instead of a notebook instance.1. read `raw/hour.csv` from S32. engineer features, write to `featured/`3. train on this instance, write the artifact to `model/`Regression: the target is a count, not a class.

## 1. SetupThe Studio distribution image already ships pandas, scikit-learn andboto3, so there is no `%pip install` step here.`BUCKET` comes from the domain stack:```terraform -chdir=infra/domain output -raw data_bucket```

In [ ]:
import io

import boto3
import pandas as pd

REGION = "ca-central-1"

# Random suffix, so this changes on every destroy/apply cycle.
BUCKET = "REPLACE-ME"

RAW_KEY = "raw/hour.csv"
FEATURED_KEY = "featured/hour.parquet"
MODEL_KEY = "model/model.joblib"
FEATURES_KEY = "model/features.joblib"

s3 = boto3.client("s3", region_name=REGION)

# Fails fast and loudly if the bucket name or the S3 grant is wrong.
s3.head_bucket(Bucket=BUCKET)
print(f"s3://{BUCKET} reachable")

## 2. Load17,379 hourly records spanning 2011-2012. `cnt` is the target.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"{len(df)} rows, {df.dteday.min()} -> {df.dteday.max()}")
df.head()

## 3. Features`casual` and `registered` sum to `cnt` in every row, so keeping themwould leak the target and produce a meaningless perfect score. `instant`is a row index and `dteday` is superseded by the calendar columns.

In [ ]:
TARGET = "cnt"
DROP = ["instant", "dteday", "casual", "registered", TARGET]
FEATURES = [c for c in df.columns if c not in DROP]

# Proof of the leak, worth seeing once.
leak = (df.casual + df.registered != df[TARGET]).sum()
print(f"rows where casual + registered != cnt: {leak}")
print(f"{len(FEATURES)} features: {FEATURES}")

## 4. Persist the feature setWriting `featured/` is what makes the prefix real. Phase 7 reads thisinstead of re-deriving the frame inside the pipeline.

In [ ]:
# yr is already in FEATURES and doubles as the split key, so the frame is
# just the features plus the target.
featured = df[FEATURES + [TARGET]]

buf = io.BytesIO()
featured.to_parquet(buf, index=False)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, FEATURED_KEY)

print(f"{featured.shape[0]} rows x {featured.shape[1]} cols")
print(f"s3://{BUCKET}/{FEATURED_KEY}")

## 5. Split by timeTrain on 2011, test on 2012. A random split would let the model seehours adjacent to the ones it is scored on, which inflates the result.

In [ ]:
train = df[df.yr == 0]
test = df[df.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")

## 6. TrainRandom forest handles the mix of categorical codes and normalizedfloats without scaling or encoding.`min_samples_leaf=5` costs ~0.7% r2 versus fully grown trees and cutsthe artifact from ~70 MB to ~12 MB, which matters on a serverlessendpoint that reloads the model on every cold start.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"rmse={rmse:.1f}  mae={mae:.1f}  r2={r2:.3f}")

## 7. What drives demand

In [ ]:
importance = sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1])

for name, score in importance[:6]:
    print(f"  {name:12} {score:.3f}  {'#' * int(score * 60)}")

## 8. Save the modelBoth objects go to `model/`. Phase 8 serves them from there.

In [ ]:
import joblib

buf = io.BytesIO()
joblib.dump(model, buf)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, MODEL_KEY)

# Column order is part of the model contract: inference must rebuild the
# feature frame in exactly this order.
buf = io.BytesIO()
joblib.dump(FEATURES, buf)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, FEATURES_KEY)

print(f"s3://{BUCKET}/{MODEL_KEY}")
print(f"s3://{BUCKET}/{FEATURES_KEY}")

## 9. Verify the round tripThe phase 4 exit check: the artifact exists in `model/` and reloads to amodel that predicts identically.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=MODEL_KEY)
reloaded = joblib.load(io.BytesIO(obj["Body"].read()))

assert np.allclose(reloaded.predict(X_test), pred), "reloaded model disagrees"
print("round trip ok -- artifact in model/ matches the in-memory model")